# 06 · Model experiments: B0, M1, M2, with COVID kept or left out

Three candidates, trained on the same features and the same train split, scored with the same metrics as the training job (`utils/model.py`):

- **B0**, the seasonal baseline: the area's mean count for the same hour of the week over the previous 4 weeks;
- **M1**, Poisson regression on the scaled features;
- **M2**, XGBoost with a Poisson objective, at the starting values in config.

Each is trained twice: with the COVID months kept (the `cal_covid_regime` flag marks them), and with them left out of train. P0 is the share of a shift's crashes that fall in the 10 areas forecast highest.

In [1]:
import os
import time

import pandas as pd
import yaml

# run from the repo root, where main.py, config/ and datamart/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from utils.model import LABEL, fairness_groups, feature_columns, fit_candidate, load_split, metrics, predict

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
features = feature_columns(config)
data = {split: load_split(split, features, config) for split in ["train", "validation", "test"]}
for split, df in data.items():
    print(f"{split:<10} {len(df):>9,} rows, {df['hour_start'].min():%Y-%m} to {df['hour_start'].max():%Y-%m}, "
          f"{df[LABEL].mean():.3f} crashes per area-hour")
print(len(features), "features:", ", ".join(features))

train      2,533,377 rows, 2018-04 to 2021-12, 0.161 crashes per area-hour
validation   334,411 rows, 2022-01 to 2022-06, 0.156 crashes per area-hour
test         340,032 rows, 2022-07 to 2022-12, 0.162 crashes per area-hour
25 features: cal_covid_regime, cal_hour, cal_is_holiday, cal_month, cal_weekday, crash_area_how_mean_4w, crash_area_notified_1h, crash_area_notified_24h, crash_area_notified_3h, crash_city_how_mean_4w, crash_city_notified_1h, crash_city_notified_24h, crash_city_notified_3h, lic_nightlife_active, lic_package_goods_active, taxi_area_trips_1h, taxi_city_trips_1h, trk_congestion_index, trk_coverage, wx_age_hours, wx_dew_point_c, wx_precip_1h_mm, wx_temp_c, wx_visibility_m, wx_wind_ms


## 1 · Train and score every candidate, both ways

Option (b) leaves the COVID months (config `covid_regime`) out of train. Validation and test don't change, so the two options are scored on the same rows.

In [2]:
covid = config["features"]["covid_regime"]


def score(artefact, df):
    return pd.DataFrame({"community_area": df["community_area"], "hour_start": df["hour_start"],
                         "actual": df[LABEL], "predicted": predict(artefact, df)})


rows, models = [], {}
for option in ["kept", "left out"]:
    train = data["train"]
    if option == "left out":
        train = train[~train["hour_start"].dt.strftime("%Y-%m").between(covid["start"], covid["end"])]
    groups = fairness_groups(train, config)
    for algorithm in ["B0", "M1", "M2"]:
        started = time.time()
        model, feature_list, pre = fit_candidate(algorithm, train, features, config)
        artefact = {"model": model, "feature_list": feature_list, "preprocessing_transformers": pre}
        models[(algorithm, option)] = artefact
        for split in ["validation", "test"]:
            m = metrics(score(artefact, data[split]), config, groups)
            rows.append({"COVID": option, "model": algorithm, "split": split, "P0": m["P0"], "MAE": m["MAE"],
                         "deviance": m["poisson_deviance"], "bias": m["bias"],
                         "P0 high taxi": m["groups"]["high_taxi_coverage"]["P0"],
                         "P0 other": m["groups"]["other_areas"]["P0"]})
        print(f"{algorithm}, COVID {option}: {len(train):,} train rows, {time.time() - started:.0f} s")
results = pd.DataFrame(rows)

B0, COVID kept: 2,533,377 train rows, 1 s
M1, COVID kept: 2,533,377 train rows, 10 s
M2, COVID kept: 2,533,377 train rows, 33 s
B0, COVID left out: 1,633,555 train rows, 0 s
M1, COVID left out: 1,633,555 train rows, 7 s
M2, COVID left out: 1,633,555 train rows, 21 s


## 2 · P0, the metric we select on

In [3]:
results.pivot_table(index="model", columns=["split", "COVID"], values="P0").round(3)

split   test          validation         
COVID   kept left out       kept left out
model                                    
B0     0.291    0.291      0.276    0.276
M1     0.311    0.310      0.300    0.299
M2     0.312    0.311      0.305    0.304

## 3 · Every metric, for the report

MAE and deviance per area-hour (lower is better); bias = forecast crashes ÷ actual crashes (1 is unbiased); P0 per area group, the fairness check.

In [4]:
results.set_index(["split", "COVID", "model"]).sort_index().round(3)

P0    MAE  deviance   bias  P0 high taxi  \
split      COVID    model                                                
test       kept     B0     0.291  0.247     1.109  1.011         0.877   
                    M1     0.311  0.247     0.567  0.959         0.951   
                    M2     0.312  0.244     0.548  0.979         0.952   
           left out B0     0.291  0.247     1.109  1.011         0.877   
                    M1     0.310  0.246     0.567  0.946         0.952   
                    M2     0.311  0.244     0.548  0.987         0.952   
validation kept     B0     0.276  0.237     1.103  0.972         0.816   
                    M1     0.300  0.240     0.556  0.942         0.939   
                    M2     0.305  0.237     0.536  0.974         0.938   
           left out B0     0.276  0.237     1.103  0.972         0.816   
                    M1     0.299  0.240     0.556  0.934         0.940   
                    M2     0.304  0.235     0.536  0.962         0.938   

                           P0 other  
split      COVID    model            
test       kept     B0        0.210  
                    M1        0.221  
                    M2        0.221  
           left out B0        0.210  
                    M1        0.220  
                    M2        0.219  
validation kept     B0        0.208  
                    M1        0.218  
                    M2        0.221  
           left out B0        0.208  
                    M1        0.216  
                    M2        0.220

## 4 · What M2 leans on

XGBoost's importance by gain: how much each feature's splits reduce the loss, as a share of the total.

In [5]:
m2 = models[("M2", "kept")]["model"]
pd.Series(m2.feature_importances_, index=features).sort_values(ascending=False).head(10).round(3)

crash_area_how_mean_4w      0.516
crash_city_how_mean_4w      0.123
lic_package_goods_active    0.090
crash_area_notified_24h     0.085
crash_city_notified_1h      0.036
taxi_area_trips_1h          0.021
cal_hour                    0.018
lic_nightlife_active        0.013
cal_covid_regime            0.012
wx_precip_1h_mm             0.011
dtype: float32